# 序列并行

- 简单来说，就是把一张 GPU 吃不下的一条超长数据（序列），切成好几段，分给多张 GPU 一起去算。

不过，为了让这个过程在数学上等同于“完整计算”，实际的技术实现比“切分+处理”要稍微精妙一些。我们可以从以下三个层面来深入理解：

1. 为什么要切？（为了解决显存爆炸）

在 Transformer 架构中，<font color='red'>注意力机制（Attention） 的计算复杂度是序列长度的平方级（ $O(N^2)$ ）。</font>
 - 如果不切：当序列长度达到 32K 或 128K 时，存储注意力矩阵和 KV 缓存所需的显存会瞬间撑爆单张显卡（比如 A100）。
 - 切分后：<font color='red'>每张显卡只需要存储和处理序列的一小部分</font>（比如 1/P），显存压力直接降低为原来的 1/P。
 
2. 怎么切？（两种主流流派）

虽然都是“拆分”，但在具体怎么拆分和协作上，主要有两种不同的技术路线：

 - 流派一：DeepSpeed Ulysses（维度转换法）
    - 切法：把序列切成 P 份，每张卡拿一份。
    - 处理：通过一种叫 All-to-All 的通信方式，把“序列维度”的信息转换到“注意力头维度”上。
    - 结果：每张卡虽然只算一部分序列，但它负责计算所有的“注意力头”。算完后再转回来。
    - 特点：通信量小，但受限于注意力头的数量（卡数不能超过头数）。
    
 - 流派二：Ring-Attention（环形接力法）
    - 切法：同样把序列切成 P 份。
    - 处理：<font color='red'>显卡们连成一个环。每张卡算自己这一块序列时，不够的信息（其他卡的 Key/Value）通过 P2P 通信像接力棒一样传过来。</font>
    - 结果：通过多轮“接收-计算-发送”，每张卡最终都能拼凑出全局的上下文信息。
    - 特点：显存利用率极致，没有卡数限制，非常适合超长文本（如 1M 上下文）。
    
    
3. 代价是什么？（用通信换显存）

正如你所说，我们是“一同处理”。但这个“一同”是有代价的。
- 通信开销：<font color='red'>因为每张卡手里只有一半的拼图，它们必须频繁地互相交换数据（通信），才能拼出完整的画面。</font>
- 核心权衡：序列并行本质上是用网络通信时间来换取显存空间。只要通信够快（比如用 NVLink），这个交换就是非常划算的。
